# Text Bias Audit

This notebook demonstrates the AEGIS text bias auditing pipeline.
It creates demographically-framed prompt pairs, computes cosine distances
between their embeddings, and scores bias levels.

**Goals:**
1. Create demographic-framed prompts
2. Compute cosine distances between embeddings
3. Score bias levels
4. Full audit across categories
5. Visualize bias scores
6. Summary

In [ ]:
# Cell 1: Imports
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np

# Project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"NumPy: {np.__version__}")

try:
    import matplotlib.pyplot as plt
    print(f"Matplotlib: {plt.matplotlib.__version__}")
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("Matplotlib not available")

from app.ml.text_bias.prompt_framer import PromptFramer, BiasCategory
from app.ml.text_bias.cosine_distance import CosineDistanceCalculator
from app.ml.text_bias.bias_scorer import TextBiasScorer, _normalize_score, _classify_bias

print("\nAEGIS text bias modules loaded successfully.")

In [ ]:
# Cell 2: Create demographic-framed prompts
framer = PromptFramer(seed=42)

print("Supported bias categories:")
for cat in BiasCategory:
    demos = framer.get_demographics(cat.value)
    print(f"  {cat.value}: {demos}")

# Generate prompt pairs for each category
print(f"\nGenerating prompt pairs...")
audit_set = framer.generate_audit_set(
    categories=None,  # All categories
    templates_per_category=3,
    stereoset_per_category=3,
)

print(f"Total audit items: {len(audit_set)}")

# Show some examples
print(f"\nExample prompt pairs:")
for item in audit_set[:6]:
    print(f"\n  Category: {item['category']} ({item['type']})")
    print(f"    A: {item['prompt_a']}")
    print(f"    B: {item['prompt_b']}")

In [ ]:
# Cell 3: Compute cosine distances between embeddings
#
# Since we don't have an actual LLM/embedding model available in this
# notebook, we use random embeddings seeded deterministically to
# demonstrate the pipeline. In production, replace with real embeddings.

rng = np.random.RandomState(42)
distance_calc = CosineDistanceCalculator()

# Generate deterministic "embeddings" for each prompt
# (In production, these would come from an embedding model)
def mock_embedding(text: str, dim: int = 128) -> np.ndarray:
    """Generate a deterministic embedding from text hash."""
    seed = hash(text) % (2**31)
    local_rng = np.random.RandomState(seed)
    vec = local_rng.randn(dim).astype(np.float64)
    vec = vec / max(np.linalg.norm(vec), 1e-10)
    return vec

EMBEDDING_DIM = 128
results = []

for item in audit_set:
    emb_a = mock_embedding(item['prompt_a'], EMBEDDING_DIM)
    emb_b = mock_embedding(item['prompt_b'], EMBEDDING_DIM)
    
    # Add systematic bias for stereoset items (to demonstrate detection)
    if item['type'] == 'stereoset':
        # Stereotype prompts get a systematic shift
        bias_shift = rng.randn(EMBEDDING_DIM) * 0.3
        emb_b = emb_b + bias_shift
        emb_b = emb_b / max(np.linalg.norm(emb_b), 1e-10)
    
    cos_dist = distance_calc.compute(emb_a, emb_b)
    cos_sim = distance_calc.compute_similarity(emb_a, emb_b)
    
    results.append({
        'category': item['category'],
        'type': item['type'],
        'prompt_a': item['prompt_a'],
        'prompt_b': item['prompt_b'],
        'cosine_distance': cos_dist,
        'cosine_similarity': cos_sim,
    })

# Show some distances
print(f"Computed distances for {len(results)} prompt pairs.")
print(f"\nSample distances:")
print(f"{'Category':<12} {'Type':<10} {'Distance':>10} {'Similarity':>10}")
print("-" * 44)
for r in results[:10]:
    print(f"{r['category']:<12} {r['type']:<10} {r['cosine_distance']:>10.6f} {r['cosine_similarity']:>10.6f}")

In [ ]:
# Cell 4: Score bias levels
scorer = TextBiasScorer()

# Score each pair
for r in results:
    score = scorer.score_pair(
        emb_a=mock_embedding(r['prompt_a'], EMBEDDING_DIM),
        emb_b=mock_embedding(r['prompt_b'], EMBEDDING_DIM),
        cosine_distance=r['cosine_distance'],
    )
    r['bias_score'] = score.normalized_score
    r['bias_level'] = score.bias_level.value

# Show scored results
print("Bias Score Distribution:")
level_counts = {}
for r in results:
    level = r['bias_level']
    level_counts[level] = level_counts.get(level, 0) + 1

for level in ['NONE', 'LOW', 'MEDIUM', 'HIGH', 'SEVERE']:
    count = level_counts.get(level, 0)
    pct = count / max(len(results), 1) * 100
    bar = '#' * int(pct / 2)
    print(f"  {level:<8} {count:>4} ({pct:>5.1f}%) {bar}")

# Show highest bias pairs
sorted_results = sorted(results, key=lambda x: -x['cosine_distance'])
print(f"\nTop 5 highest bias pairs:")
for r in sorted_results[:5]:
    print(f"  [{r['bias_level']}] d={r['cosine_distance']:.4f}, score={r['bias_score']:.1f}")
    print(f"    A: {r['prompt_a'][:80]}")
    print(f"    B: {r['prompt_b'][:80]}")

In [ ]:
# Cell 5: Full audit across categories
print("Running full dataset-level audit...")

# Prepare flat results for scorer
flat_results = [
    {
        'cosine_distance': r['cosine_distance'],
        'category': r['category'],
        'normalized_score': r['bias_score'],
    }
    for r in results
]

summary = scorer.score_dataset(flat_results)

print(f"\nDataset Bias Summary:")
print(f"  Total pairs:      {summary.total_pairs}")
print(f"  Mean distance:    {summary.mean_distance:.6f}")
print(f"  Median distance:  {summary.median_distance:.6f}")
print(f"  Max distance:     {summary.max_distance:.6f}")
print(f"  Min distance:     {summary.min_distance:.6f}")
print(f"  Std distance:     {summary.std_distance:.6f}")
print(f"  Bias index:       {summary.bias_index:.2f} / 100")

print(f"\nPer-Category Breakdown:")
print(f"{'Category':<12} {'Count':>6} {'Mean Dist':>10} {'Bias Index':>11}")
print("-" * 41)
for cat, stats in sorted(summary.per_category.items()):
    print(f"{cat:<12} {stats['count']:>6} {stats['mean']:>10.6f} {stats['bias_index']:>11.2f}")

print(f"\nRecommendations:")
for rec in summary.recommendations:
    print(f"  - {rec}")

In [ ]:
# Cell 6: Visualize bias scores
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Plot 1: Bias scores by category
    categories = list(summary.per_category.keys())
    bias_indices = [summary.per_category[c]['bias_index'] for c in categories]
    mean_dists = [summary.per_category[c]['mean'] for c in categories]
    
    colors = ['green' if bi < 30 else 'orange' if bi < 60 else 'red' for bi in bias_indices]
    axes[0].barh(categories, bias_indices, color=colors, alpha=0.7)
    axes[0].axvline(x=50, color='red', linestyle='--', label='High bias threshold', alpha=0.5)
    axes[0].set_xlabel('Bias Index (0-100)')
    axes[0].set_title('Per-Category Bias Index')
    axes[0].legend(fontsize=8)
    
    # Plot 2: Distance distribution histogram
    distances = [r['cosine_distance'] for r in results]
    axes[1].hist(distances, bins=20, color='steelblue', alpha=0.7, edgecolor='white')
    axes[1].axvline(x=0.1, color='green', linestyle='--', label='LOW threshold', alpha=0.7)
    axes[1].axvline(x=0.3, color='orange', linestyle='--', label='MEDIUM threshold', alpha=0.7)
    axes[1].axvline(x=0.6, color='red', linestyle='--', label='HIGH threshold', alpha=0.7)
    axes[1].set_xlabel('Cosine Distance')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Distance Distribution')
    axes[1].legend(fontsize=8)
    
    # Plot 3: Template vs StereoSet comparison
    template_dists = [r['cosine_distance'] for r in results if r['type'] == 'template']
    stereoset_dists = [r['cosine_distance'] for r in results if r['type'] == 'stereoset']
    
    if template_dists:
        axes[2].boxplot([template_dists, stereoset_dists], 
                        labels=['Template', 'StereoSet'],
                        patch_artist=True)
        axes[2].set_ylabel('Cosine Distance')
        axes[2].set_title('Template vs StereoSet Distances')
    
    plt.suptitle(f'Text Bias Audit Results (Bias Index: {summary.bias_index:.1f}/100)', 
                 fontsize=14)
    plt.tight_layout()
    plt.show()
    print("Bias visualizations rendered above.")
else:
    print("Matplotlib not available. Skipping visualization.")

In [ ]:
# Cell 7: Summary
print("=" * 60)
print("Text Bias Audit - Summary")
print("=" * 60)

print(f"""
Audit Setup:
  - Prompt framer with {len(BiasCategory)} bias categories
  - Template pairs + StereoSet pairs per category
  - Total prompt pairs: {summary.total_pairs}

Overall Results:
  - Bias index: {summary.bias_index:.2f} / 100
  - Mean cosine distance: {summary.mean_distance:.6f}
  - Score distribution: {dict(summary.score_distribution)}

Per-Category:
  - Best (least biased): {min(summary.per_category.items(), key=lambda x: x[1]['bias_index'])[0]}
  - Worst (most biased): {max(summary.per_category.items(), key=lambda x: x[1]['bias_index'])[0]}

Key Takeaways:
  1. The PromptFramer generates demographically-balanced prompt pairs
     across 5 bias categories.
  2. Cosine distance measures embedding divergence caused by demographic swaps.
  3. The TextBiasScorer maps distances to human-readable bias levels.
  4. In production, use the full TextAuditor with an actual LLM to get
     real responses and embeddings.

Production Notes:
  - Use TextAuditor class with a real LLM (OpenAI, Anthropic, or local).
  - The audit set can be customized with domain-specific templates.
  - Results are serializable and can be exported via the API.
""")